# Data Accessibility Explorer

**Problem**: How can we make the mirrored Regulations.gov dataset easier to explore and analyze for researchers and non-technical stakeholders?

**Stakeholder Quotes:**
- "Accessing all the data in this format can be slow, but it's way better than Regulations.gov."
- "What can we build on top of the raw Mirrulations data?"

This notebook provides:
- Quick data exploration tools
- Schema documentation
- Sample queries for common tasks
- Export utilities

In [1]:
import duckdb
import pandas as pd

import os
R2_BASE_URL = (
    os.environ.get("SPICY_REGS_R2_URL")
    or os.environ.get("R2_PUBLIC_URL")
    or f"https://data.{os.environ.get('SPICYREGS_DOMAIN') or 'spicy-regs.dev'}"
).rstrip("/")

conn = duckdb.connect()
conn.execute("INSTALL httpfs; LOAD httpfs;")
conn.execute("PRAGMA disable_progress_bar")
print("✓ DuckDB ready - querying Parquet files from R2")

✓ DuckDB ready - querying Parquet files from R2


## Data Overview

In [2]:
import urllib.request


def _remote_size_mb(url: str) -> float:
    """Content-Length via HEAD — no body download."""
    req = urllib.request.Request(url, method="HEAD", headers={"User-Agent": "spicy-regs-notebook"})
    with urllib.request.urlopen(req) as resp:
        return int(resp.headers.get("Content-Length", 0)) / (1024 * 1024)


# Get dataset statistics. Queries stream straight from R2 — no local download.
# (comments.parquet alone is ~2.5 GB; downloading it to a local ../data/ cache
# is slow on first run *and* a trap afterwards, since a cached copy never
# picks up schema changes or new rows unless you remember to delete it.)
print("📊 Dataset Statistics")
print("=" * 50)

for data_type in ["dockets", "documents", "comments"]:
    url = f"{R2_BASE_URL}/{data_type}.parquet"
    size_mb = _remote_size_mb(url)

    stats = conn.execute(f"""
        SELECT 
            COUNT(*) as rows,
            COUNT(DISTINCT agency_code) as agencies
        FROM read_parquet('{url}')
    """).fetchone()
    print(f"  {data_type}: {stats[0]:,} rows, {stats[1]} agencies, {size_mb:.1f} MB")

📊 Dataset Statistics


  dockets: 276,326 rows, 196 agencies, 13.7 MB


  documents: 1,992,481 rows, 316 agencies, 52.2 MB


  comments: 25,558,418 rows, 179 agencies, 2461.9 MB


In [3]:
# Extended dataset statistics
print("\n📊 Extended Statistics")
print("=" * 50)

# Date range
date_range = conn.execute(f"""
    SELECT 
        MIN(posted_date) as earliest,
        MAX(posted_date) as latest
    FROM read_parquet('{R2_BASE_URL}/comments.parquet')
    WHERE posted_date IS NOT NULL
""").fetchone()
print(f"Comment date range: {date_range[0][:10]} to {date_range[1][:10]}")

# Top agencies by comment volume
print("\nTop 5 agencies by comment volume:")
top_agencies = conn.execute(f"""
    SELECT agency_code, COUNT(*) as comments
    FROM read_parquet('{R2_BASE_URL}/comments.parquet')
    GROUP BY agency_code
    ORDER BY comments DESC
    LIMIT 5
""").fetchall()
for agency, count in top_agencies:
    print(f"  {agency}: {count:,}")

# Average comments per docket
avg_comments = conn.execute(f"""
    SELECT ROUND(AVG(cnt), 1) as avg_comments
    FROM (
        SELECT docket_id, COUNT(*) as cnt
        FROM read_parquet('{R2_BASE_URL}/comments.parquet')
        GROUP BY docket_id
    )
""").fetchone()[0]
print(f"\nAverage comments per docket: {avg_comments}")

# Dockets with open comment periods (future end dates, relative to today —
# dates are stored as strings, so cast before comparing to CURRENT_DATE)
open_dockets = conn.execute(f"""
    SELECT COUNT(DISTINCT docket_id)
    FROM read_parquet('{R2_BASE_URL}/documents.parquet')
    WHERE TRY_CAST(comment_end_date AS DATE) > CURRENT_DATE
""").fetchone()[0]
print(f"Dockets with open comment periods: {open_dockets}")


📊 Extended Statistics


Comment date range: 0000-12-30 to 2026-08-12

Top 5 agencies by comment volume:


  FWS: 2,629,145
  FDA: 1,801,725
  CMS: 1,416,223
  EPA: 1,135,527
  HHS: 1,105,806



Average comments per docket: 428.5


Dockets with open comment periods: 817


## Schema Reference

In [4]:
# Show schema for each dataset
for data_type in ["dockets", "documents", "comments"]:
    url = f"{R2_BASE_URL}/{data_type}.parquet"
    schema = conn.execute(f"DESCRIBE SELECT * FROM read_parquet('{url}')").fetchdf()
    print(f"\n📋 {data_type.upper()} Schema:")
    print("-" * 40)
    for _, row in schema.iterrows():
        print(f"  {row['column_name']}: {row['column_type']}")


📋 DOCKETS Schema:
----------------------------------------
  docket_id: VARCHAR
  agency_code: VARCHAR
  title: VARCHAR
  docket_type: VARCHAR
  modify_date: VARCHAR
  abstract: VARCHAR
  rin: VARCHAR



📋 DOCUMENTS Schema:
----------------------------------------
  document_id: VARCHAR
  docket_id: VARCHAR
  agency_code: VARCHAR
  title: VARCHAR
  document_type: VARCHAR
  posted_date: VARCHAR
  modify_date: VARCHAR
  comment_start_date: VARCHAR
  comment_end_date: VARCHAR
  file_url: VARCHAR
  attachments_json: VARCHAR
  fr_doc_num: VARCHAR
  withdrawn: VARCHAR
  reason_withdrawn: VARCHAR
  additional_rins: VARCHAR
  text_content: VARCHAR
  text_extraction_status: VARCHAR



📋 COMMENTS Schema:
----------------------------------------
  comment_id: VARCHAR
  docket_id: VARCHAR
  agency_code: VARCHAR
  first_name: VARCHAR
  last_name: VARCHAR
  organization: VARCHAR
  category: VARCHAR
  title: VARCHAR
  comment: VARCHAR
  document_type: VARCHAR
  posted_date: VARCHAR
  modify_date: VARCHAR
  receive_date: VARCHAR
  attachments_json: VARCHAR
  text_content: VARCHAR
  text_extraction_status: VARCHAR


## Quick Search

In [5]:
def search_dockets(keyword, agency=None, limit=20):
    """Search dockets by keyword in title."""
    agency_filter = f"AND agency_code = '{agency}'" if agency else ""
    return conn.execute(f"""
        SELECT docket_id, agency_code, title, docket_type, modify_date
        FROM read_parquet('{R2_BASE_URL}/dockets.parquet')
        WHERE LOWER(title) LIKE '%{keyword.lower()}%'
        {agency_filter}
        ORDER BY modify_date DESC
        LIMIT {limit}
    """).fetchdf()

# Example: search for climate dockets
search_dockets("climate", limit=10)

,docket_id,agency_code,title,docket_type,modify_date
0,FAR-2021-0015,FAR,Federal Acquisition Regulation: Disclosure of ...,Rulemaking,2026-01-06T15:15:26Z
1,FAR-2021-0016,FAR,Federal Acquisition Regulation: Minimizing th...,Rulemaking,2026-01-06T15:15:14Z
2,EPA-HQ-OAR-2025-0162,EPA,Extension of Deadlines in Standards of Perform...,Rulemaking,2025-12-03T15:03:57Z
3,EPA-HQ-OAR-2024-0358,EPA,Reconsideration of Oil and Natural Gas Sector ...,Rulemaking,2025-10-10T17:05:28Z
4,DOE-HQ-2025-0207,DOE,A Critical Review of Impacts of Greenhouse Gas...,Rulemaking,2025-10-07T14:00:20Z
5,DOD-2025-OS-0342,DOD,Defense Organizational Climate Survey (DEOCS);...,Nonrulemaking,2025-07-29T12:47:52Z
6,EPA-HQ-OAR-2021-0317,EPA,"Standards of Performance for New, Reconstructe...",Rulemaking,2025-07-23T17:07:06Z
7,EPA-HQ-OAR-2023-0492,EPA,Transportation and Climate Division (TCD) Gran...,Nonrulemaking,2025-07-21T11:05:58Z
8,FAA-2025-0289,FAA,Climate Systems NW LLC - Exemption/Rulemaking,Nonrulemaking,2025-07-07T18:43:24Z
9,PTO-P-2025-0010,PTO,Termination of the Climate Change Mitigation P...,Nonrulemaking,2025-04-18T14:16:49Z


In [6]:
def get_docket_comments(docket_id, limit=100):
    """Get comments for a specific docket."""
    return conn.execute(f"""
        SELECT comment_id, title, posted_date, LEFT(comment, 200) as preview
        FROM read_parquet('{R2_BASE_URL}/comments.parquet')
        WHERE docket_id = '{docket_id}'
        ORDER BY posted_date DESC
        LIMIT {limit}
    """).fetchdf()

# Example
get_docket_comments("EPA-HQ-OAR-2021-0317", limit=5)

,comment_id,title,posted_date,preview
0,EPA-HQ-OAR-2021-0317-4068,Comment submitted by Interstate Natural Gas As...,2025-07-23T04:00:00Z,See Attached
1,EPA-HQ-OAR-2021-0317-4067,Supplemental Letter to Petition for Reconsider...,2024-10-23T04:00:00Z,See Attached
2,EPA-HQ-OAR-2021-0317-4066,Comment submitted by Air Quality Division (AQD...,2024-09-09T04:00:00Z,The Wyoming Department of Environmental Qualit...
3,EPA-HQ-OAR-2021-0317-4065,Comment submitted by American Petroleum Instit...,2024-09-04T04:00:00Z,The American Petroleum Institute (API) respect...
4,EPA-HQ-OAR-2021-0317-4063,"Comment submitted by Lumina Sky Consulting, PC",2024-08-28T04:00:00Z,Please see attached comments.


## Agency Explorer

In [7]:
# List all agencies with activity
agencies = conn.execute(f"""
    SELECT 
        agency_code,
        COUNT(*) as docket_count
    FROM read_parquet('{R2_BASE_URL}/dockets.parquet')
    GROUP BY agency_code
    ORDER BY docket_count DESC
""").fetchdf()

print(f"Found {len(agencies)} agencies with dockets:")
agencies.head(30)

Found 196 agencies with dockets:


,agency_code,docket_count
0,FDA,64140
1,FAA,51339
2,EPA,21443
3,USCG,13536
4,DOT,12033
5,FMCSA,8454
6,PHMSA,8134
7,NRC,5572
8,NOAA,5188
9,NHTSA,4606


## Export Utilities

In [8]:
def export_to_csv(query, filename):
    """Export query results to CSV."""
    df = conn.execute(query).fetchdf()
    df.to_csv(filename, index=False)
    print(f"Exported {len(df)} rows to {filename}")
    return df

# Example: export EPA dockets to CSV
# export_to_csv(f"""
#     SELECT * FROM read_parquet('{R2_BASE_URL}/dockets.parquet')
#     WHERE agency_code = 'EPA'
# """, "epa_dockets.csv")

In [9]:
def export_docket_summary(docket_id, output_dir="."):
    """Export a complete docket analysis package."""
    import os
    
    # Docket info
    docket = conn.execute(f"""
        SELECT * FROM read_parquet('{R2_BASE_URL}/dockets.parquet')
        WHERE docket_id = '{docket_id}'
    """).fetchdf()
    
    # Documents
    docs = conn.execute(f"""
        SELECT * FROM read_parquet('{R2_BASE_URL}/documents.parquet')
        WHERE docket_id = '{docket_id}'
    """).fetchdf()
    
    # Comments
    comments = conn.execute(f"""
        SELECT * FROM read_parquet('{R2_BASE_URL}/comments.parquet')
        WHERE docket_id = '{docket_id}'
    """).fetchdf()
    
    # Save
    docket.to_csv(f"{output_dir}/{docket_id}_docket.csv", index=False)
    docs.to_csv(f"{output_dir}/{docket_id}_documents.csv", index=False)
    comments.to_csv(f"{output_dir}/{docket_id}_comments.csv", index=False)
    
    print(f"Exported {docket_id}:")
    print(f"  - Docket info: 1 row")
    print(f"  - Documents: {len(docs)} rows")
    print(f"  - Comments: {len(comments)} rows")

# Example:
# export_docket_summary("EPA-HQ-OAR-2021-0317")

## Common Query Templates

In [10]:
print("""
📝 Common Query Templates
========================

# 1. Search dockets by keyword
SELECT * FROM read_parquet('{url}/dockets.parquet')
WHERE LOWER(title) LIKE '%keyword%'

# 2. Get comments for a docket
SELECT * FROM read_parquet('{url}/comments.parquet')
WHERE docket_id = 'AGENCY-YEAR-NNNN'

# 3. Find dockets with open comment periods
SELECT * FROM read_parquet('{url}/documents.parquet')
WHERE TRY_CAST(comment_end_date AS DATE) > CURRENT_DATE

# 4. Count comments by agency
SELECT agency_code, COUNT(*) FROM read_parquet('{url}/comments.parquet')
GROUP BY agency_code ORDER BY 2 DESC

# 5. Find duplicate comments
SELECT comment, COUNT(*) as cnt
FROM read_parquet('{url}/comments.parquet')
WHERE docket_id = 'XXX'
GROUP BY comment HAVING cnt > 1
""".format(url=R2_BASE_URL))


📝 Common Query Templates

# 1. Search dockets by keyword
SELECT * FROM read_parquet('https://data.spicy-regs.dev/dockets.parquet')
WHERE LOWER(title) LIKE '%keyword%'

# 2. Get comments for a docket
SELECT * FROM read_parquet('https://data.spicy-regs.dev/comments.parquet')
WHERE docket_id = 'AGENCY-YEAR-NNNN'

# 3. Find dockets with open comment periods
SELECT * FROM read_parquet('https://data.spicy-regs.dev/documents.parquet')
WHERE TRY_CAST(comment_end_date AS DATE) > CURRENT_DATE

# 4. Count comments by agency
SELECT agency_code, COUNT(*) FROM read_parquet('https://data.spicy-regs.dev/comments.parquet')
GROUP BY agency_code ORDER BY 2 DESC

# 5. Find duplicate comments
SELECT comment, COUNT(*) as cnt
FROM read_parquet('https://data.spicy-regs.dev/comments.parquet')
WHERE docket_id = 'XXX'
GROUP BY comment HAVING cnt > 1

